> 在线讲义：[第 2 章 · 梯度与反向传播：“往猜得更对的方向挪”到底怎么算](https://wikieden.github.io/beyond-transformer/lessons/02-backprop.html) · 本书《超越 Transformer》

# 第 2 章 · 梯度与反向传播：“往猜得更对的方向挪”到底怎么算

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/02-backprop.html) · 仓库里 `beyond-transformer/notes/lessons/02-backprop.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 斜率 | 中心差分 | `slope(f, x)` | 2.0000、4.0000 |
| 第二步 链式法则 | 斜率连乘 | `chain_example(1.0)` | 24 |
| 第三步 前向 | a、h、o、p、L | `forward()` | L = 0.1429 |
| 第三、四步 反向 | p − y、外积 | `backward()` | ∂L/∂W₁ 第 2 行 [−0.1491, −0.2981] |
| 第五步 梯度检查 | 挪 ±ε | `grad_check()` | 相对误差 4.6e-11 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `slope`**：中心差分：(f(x + h) − f(x − h)) / 2h，导数的数值近似。

In [ ]:
def slope(f, x, h=1e-4):
    """中心差分：(f(x + h) − f(x − h)) / 2h，导数的数值近似。"""
    return (f(x + h) - f(x - h)) / (2 * h)

**函数 `chain_example`**：y = (3x + 1)²：外层 u² 的导数 2u，乘内层 3x + 1 的导数 3。

In [ ]:
def chain_example(x):
    """y = (3x + 1)²：外层 u² 的导数 2u，乘内层 3x + 1 的导数 3。"""
    u = 3 * x + 1
    return u**2, 2 * u * 3

In [ ]:
X = np.array([1.0, 2.0])                       # 输入
W1 = np.array([[0.5, -1.0], [1.0, 0.5]])       # 第 1 层：h = relu(W1 x)
W2 = np.array([[1.0, -1.0], [0.5, 1.0], [-1.0, 0.0]])  # 第 2 层：o = W2 h，3 个类
Y = 1                                          # 正确类是第 1 类（从 0 数）

**函数 `softmax`**：小工具。

In [ ]:
def softmax(o):
    e = np.exp(o - o.max())
    return e / e.sum()

**函数 `forward`**：小工具。

In [ ]:
def forward(x=X, w1=W1, w2=W2, y=Y):
    a = w1 @ x
    h = np.maximum(a, 0)
    o = w2 @ h
    p = softmax(o)
    return {"a": a, "h": h, "o": o, "p": p, "loss": float(-np.log(p[y]))}

**函数 `backward`**：从 loss 往回：do = p − onehot(y)；dW2 = do hᵀ；dh = W2ᵀ do；da = dh ⊙ [a > 0]；dW1 = da xᵀ。

In [ ]:
def backward(x=X, w1=W1, w2=W2, y=Y):
    """从 loss 往回：do = p − onehot(y)；dW2 = do hᵀ；dh = W2ᵀ do；da = dh ⊙ [a > 0]；dW1 = da xᵀ。"""
    c = forward(x, w1, w2, y)
    do = c["p"].copy()
    do[y] -= 1
    dw2 = np.outer(do, c["h"])
    dh = w2.T @ do
    da = dh * (c["a"] > 0)
    dw1 = np.outer(da, x)
    return {"do": do, "dW2": dw2, "dh": dh, "da": da, "dW1": dw1}

**函数 `grad_check`**：把 W1 每个元素各挪 ±eps，用数值斜率和反向传播的结果比较，返回最大相对误差。

In [ ]:
def grad_check(eps=1e-5):
    """把 W1 每个元素各挪 ±eps，用数值斜率和反向传播的结果比较，返回最大相对误差。"""
    g = backward()["dW1"]
    num = np.zeros_like(W1)
    for i in range(2):
        for j in range(2):
            wp, wm = W1.copy(), W1.copy()
            wp[i, j] += eps
            wm[i, j] -= eps
            num[i, j] = (forward(w1=wp)["loss"] - forward(w1=wm)["loss"]) / (2 * eps)
    return float(np.max(np.abs(num - g) / np.maximum(1e-8, np.abs(num) + np.abs(g)))), num

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 导数 = 斜率")
for x in (1.0, 2.0):
    print(f"  f(x) = x² 在 x = {x}：数值斜率 {slope(lambda t: t * t, x):.4f}（公式 2x = {2 * x}）")
print("\n第二步 链式法则")
y, d = chain_example(1.0)
print(f"  y = (3x + 1)²，x = 1：y = {y}，dy/dx = 2·4·3 = {d}，数值斜率 {slope(lambda t: chain_example(t)[0], 1.0):.4f}")

In [ ]:
print("\n第三步 两层小网络的前向")
c = forward()
for k in ("a", "h", "o", "p"):
    print(f"  {k} = {(np.round(c[k], 4) + 0.0).tolist()}")
print(f"  loss = −log p[1] = {c['loss']:.4f}")

In [ ]:
print("\n第四步 反向传播")
b = backward()
for k in ("do", "dW2", "dh", "da", "dW1"):
    print(f"  {k} = {(np.round(b[k], 4) + 0.0).tolist()}")

In [ ]:
print("\n第五步 梯度检查")
err, num = grad_check()
print(f"  数值梯度 dW1 = {(np.round(num, 4) + 0.0).tolist()}；最大相对误差 {err:.2e}")

## 3. 代码里才看得到的

- **反向要用前向存下的 a、h、p**：`backward()` 先调一次 `forward()` 拿中间结果。这就是训练比推理吃显存的原因。
- **梯度检查用相对误差**：梯度本身很小时，绝对误差看不出问题；分母加 1e-8 防止两边都是 0 时除零。

## 4. 自己改一改

- 把 x 改成 [3, 1]，哪些单元被 ReLU 关掉？∂L/∂W₁ 哪一行变成 0？
- 把 `grad_check` 的 eps 改成 1e-2 和 1e-9，相对误差怎么变？为什么太大、太小都不好？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 迈一小步看高低，斜率就是导数；所有偏导排成向量，就是梯度。
> 逆着梯度挪一步，乘上学习率。
> 函数套函数，斜率连着乘。
> 前向存中间，反向乘回去：下游梯度乘本层斜率，一次算完所有参数。
> softmax 接交叉熵，梯度就是 p 减 y。
> 数值斜率查小例，真训练靠自动求导。